# 04 — Customer Segmentation

## 1. Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("telco_cleaned.csv")
display(df.head())
print(df.shape)

## 2. Prepare clustering features

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Numeric behavioral/value variables are preferred for customer segmentation
candidate_cols = [
    "Tenure Months", "Monthly Charge", "Total Charges",
    "Total Revenue", "CLTV", "Churn Score", "Satisfaction Score"
]

cluster_cols = [c for c in candidate_cols if c in df.columns]
if len(cluster_cols) < 2:
    raise ValueError("Not enough numeric segmentation columns found.")

X = df[cluster_cols].apply(pd.to_numeric, errors="coerce")
X = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X), columns=cluster_cols)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Clustering features:", cluster_cols)

## 3. Choose number of clusters with elbow method

In [ ]:
from sklearn.cluster import KMeans

inertias = []
k_values = range(2, 9)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8,5))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

## 4. Silhouette analysis

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_results = []
for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    silhouette_results.append({
        "k": k,
        "silhouette_score": silhouette_score(X_scaled, labels)
    })

silhouette_df = pd.DataFrame(silhouette_results)
display(silhouette_df)

# Select the k with the highest silhouette score as a transparent data-driven choice
best_k = int(silhouette_df.loc[silhouette_df["silhouette_score"].idxmax(), "k"])
print("Selected k:", best_k)

## 5. Fit final segmentation

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
df["Customer_Segment"] = kmeans.fit_predict(X_scaled)

display(df["Customer_Segment"].value_counts().sort_index())
display(df.groupby("Customer_Segment")[cluster_cols].mean().round(2))

## 6. Visualize segments

In [ ]:
if {"Tenure Months", "Monthly Charge"}.issubset(df.columns):
    plt.figure(figsize=(8,6))
    sns.scatterplot(
        data=df,
        x="Tenure Months",
        y="Monthly Charge",
        hue="Customer_Segment",
        palette="tab10"
    )
    plt.title("Customer Segments")
    plt.show()

## 7. Save segmented data

In [ ]:
df.to_csv("telco_customer_segments.csv", index=False)
print("Saved: telco_customer_segments.csv")